# Path D-Hybrid KG Corruption GAN

End-to-end pipeline that loads a knowledge graph (dummy or FB15k-237-format),
trains a triple-level corruption GAN, and generates diverse `change_relation`
edits at inference time.

**Pipeline steps:**
1. Setup (Colab mount + imports)
2. Load knowledge graph
3. Visualise the KG
4. Build training pairs (TRIC corruption -> triple-level pairs)
5. Inspect a training sample
6. Define model (TripleGenerator + TripleDiscriminator)
7. Sanity check
8. Train (alternating G/D)
9. Training diagnostics
10. Generate K corrupted KGs (two-stage inference)
11. Operation-mix report
12. Visualise GAN vs rule output
13. Distribution validation


## Step 1: Setup

In [ ]:
from google.colab import drive
import os, sys
from pathlib import Path

drive.mount('/content/drive')

# project_path = "/content/drive/MyDrive/Flinders_NN/research_synthetic_anomaly"
project_path = "/content/drive/Othercomputers/My laptop/research_synthetic_anomaly"
# project_path = Path.cwd()

os.chdir(project_path)
if project_path not in sys.path:
    sys.path.insert(0, project_path)
print('Working directory:', os.getcwd())


In [ ]:
%pip install torchinfo torch_geometric

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data_utils
import matplotlib.pyplot as plt

random_seed = 42
np.random.seed(random_seed)
torch.manual_seed(random_seed)

compute_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {compute_device}')
print(f'torch version: {torch.__version__}')


## Step 2: Load knowledge graph

Loads a TSV-formatted KG (FB15k-237 compatible). The `kg` object is everything
downstream code needs: adjacency tensor, vocabularies, display names, types.

To switch from dummy to FB15k-237 later, change the path here. No other code
needs to change.


In [ ]:
from kg_data import load_kg

kg = load_kg('datasets/dummy_kg')

print(f"Dataset:    {kg.dataset_name}")
print(f"Entities:   {kg.num_nodes}")
print(f"Relations:  {kg.num_relations}")
print(f"Triples:    {len(kg.triples_idx)} (train), "
      f"{len(kg.valid_triples)} (valid), {len(kg.test_triples)} (test)")

# Aliases the rest of the pipeline expects (triple-level - no adjacency tensor)
clean_triples_idx_list = list(kg.triples_idx)
num_nodes              = kg.num_nodes
num_channels           = kg.num_relations
node_labels            = kg.node_labels
node_types             = kg.node_types
relation_names         = kg.relation_names

# Show the first few index-triples to confirm the loader is producing what we expect
print()
print("First 3 triples (h_idx, r_idx, t_idx) and their human-readable form:")
for h, r, t in clean_triples_idx_list[:3]:
    print(f"  ({h:2d}, {r}, {t:2d})  =>  {node_labels[h]} --{relation_names[r]}--> {node_labels[t]}")


## Step 3: Visualise the loaded KG


In [ ]:
from visualisation import visualise_kg
from kg_data import triples_to_adjacency_tensor

# Visualisation builds adjacency on-demand from the triple list.
# (For FB15k-237 scale we wouldn't visualise the whole graph anyway.)
adj_for_plot = triples_to_adjacency_tensor(
    kg.triples, kg.entity_id_to_row, kg.relation_id_to_channel,
)

visualise_kg(
    adj_for_plot, node_labels, node_types, relation_names,
    title=f"Knowledge Graph: {kg.dataset_name}",
    figsize=(7, 5),
)


## Step 4: Build training pairs

Apply TRIC `change_relation` corruption to the clean KG many times, then
extract triple-level (clean_triple, target_triple) pairs from each
(clean_graph, corrupted_graph) pair.

To use multi-operation TRIC, change `op_weights` (e.g. `{'change_relation': 2,
'swap_subject_same_type': 1, ...}`).


In [ ]:
import functools
from corruption_strategies import apply_tric_corruption
from dataset_builders import build_triple_pair_dataset

num_corruptions    = 2
num_training_pairs = 4000

# Pre-bind TRIC's kwargs. Triple-based signature returns (corrupted, edits).
tric_fn = functools.partial(
    apply_tric_corruption,
    num_entities=num_nodes,
    num_relations=num_channels,
    node_types=node_types,
    op_weights={'change_relation': 1},
)

dataset_rng = np.random.default_rng(random_seed)

# Build the (M, 6) triple-pair tensor directly. No (N, N, R) intermediate.
triple_pairs_tensor = build_triple_pair_dataset(
    clean_triples_idx_list,
    num_pairs=num_training_pairs,
    num_corruptions=num_corruptions,
    rng=dataset_rng,
    num_entities=num_nodes,
    num_relations=num_channels,
    corruption_fn=tric_fn,
    permute_entities=True,
)

clean_triples_idx     = triple_pairs_tensor[:, :3]
corrupted_triples_idx = triple_pairs_tensor[:, 3:]

print(f"Triple-pair rounds:   {num_training_pairs}")
print(f"Triple pairs built:   {len(triple_pairs_tensor)}  "
      f"(avg {len(triple_pairs_tensor) / max(num_training_pairs, 1):.2f} per round; "
      f"expected ~{num_corruptions})")

# Sanity: pure change_relation pairs preserve h and t
chk = triple_pairs_tensor.numpy()
if len(chk) > 0:
    same_h = (chk[:, 0] == chk[:, 3]).mean()
    same_t = (chk[:, 2] == chk[:, 5]).mean()
    diff_r = (chk[:, 1] != chk[:, 4]).mean()
    print(f"change_relation sanity: same_h={same_h:.3f}, same_t={same_t:.3f}, "
          f"diff_r={diff_r:.3f}  (all should be 1.0)")

training_batch_size = 64
triple_pair_dataset = data_utils.TensorDataset(clean_triples_idx, corrupted_triples_idx)
paired_dataloader   = data_utils.DataLoader(
    triple_pair_dataset, batch_size=training_batch_size, shuffle=True, drop_last=True,
)
print(f"Dataloader:  {len(triple_pair_dataset)} pairs, {len(paired_dataloader)} batches/epoch")


## Step 5: Inspect a training sample


In [ ]:
from visualisation import plot_dataset_sample

# Pick one (clean, target) training pair and visualise both as small graphs.
# We embed each pair inside a fresh view of the full KG (everything else unchanged)
# so the plotter has a sensible (N, N, R) tensor to draw from.
example_idx = 0
c_tri = tuple(triple_pairs_tensor[example_idx, :3].tolist())
t_tri = tuple(triple_pairs_tensor[example_idx, 3:].tolist())


def _index_triples_to_adj(idx_triples, n, r):
    a = np.zeros((n, n, r), dtype=np.float32)
    for h, rr, t in idx_triples:
        if h != t:
            a[h, t, rr] = 1.0
    return a


# Clean view = the full KG; corrupted view = full KG with the one edit applied
clean_view = list(kg.triples_idx)
corr_view  = [tri for tri in kg.triples_idx if tri != c_tri] + [t_tri]

clean_nxnxr = _index_triples_to_adj(clean_view, num_nodes, num_channels)
corr_nxnxr  = _index_triples_to_adj(corr_view,  num_nodes, num_channels)

plot_dataset_sample(
    clean_nxnxr, corr_nxnxr,
    node_labels, node_types, relation_names,
    num_corruptions=1,   # we visualise one substitution
)

print(f"Substitution:  ({node_labels[c_tri[0]]}, {relation_names[c_tri[1]]}, "
      f"{node_labels[c_tri[2]]})  =>  ({node_labels[t_tri[0]]}, "
      f"{relation_names[t_tri[1]]}, {node_labels[t_tri[2]]})")


## Step 6: Define the model

**Data representation:** Dai 2020 - learnable `E`, `R` embedding matrices;
triple representation `[h_emb; r_emb; t_emb]`.

**Generator:** Dai's CNN + multi-head self-attention encoder, bottleneck with
latent `z` injection, then **three independent softmax heads** (head_h, head_r,
head_t) - the structural innovation that unlocks `change_relation`. Output is
made differentiable through Gumbel-Softmax (Dai 2024).

**Discriminator:** Pix2Pix-style cGAN critic - small MLP with spectral_norm
on every linear layer, conditioned on the clean triple.


In [ ]:
# === Architecture hyperparameters ===
embedding_dim       = 100
latent_dim          = 8
encoder_base_depth  = 32
encoder_n_layers    = 3
encoder_n_heads     = 4
hidden_dim          = 128
gumbel_tau_start    = 1.0
gumbel_tau_end      = 0.5
leaky_relu_negative_slope = 0.2


class EntityEmbedding(nn.Module):
    """Dai 2020 entity embedding lookup. Trainable end-to-end."""
    def __init__(self, num_entities, d):
        super().__init__()
        self.embed = nn.Embedding(num_entities, d)
        nn.init.normal_(self.embed.weight, mean=0.0, std=1.0 / (d ** 0.5))
    def forward(self, idx):
        return self.embed(idx)
    def lookup_weight(self):
        return self.embed.weight


class RelationEmbedding(nn.Module):
    """Dai 2020 relation embedding lookup."""
    def __init__(self, num_relations, k):
        super().__init__()
        self.embed = nn.Embedding(num_relations, k)
        nn.init.normal_(self.embed.weight, mean=0.0, std=1.0 / (k ** 0.5))
    def forward(self, idx):
        return self.embed(idx)
    def lookup_weight(self):
        return self.embed.weight


def gumbel_softmax_sample(logits, tau=1.0, hard=False, eps=1e-10):
    """Gumbel-Softmax (Dai 2024) - differentiable categorical sample."""
    u = torch.rand_like(logits).clamp_(eps, 1.0 - eps)
    g = -torch.log(-torch.log(u))
    y = (logits + g) / tau
    soft = torch.softmax(y, dim=-1)
    if hard:
        idx = soft.argmax(dim=-1, keepdim=True)
        hard_one = torch.zeros_like(soft).scatter_(-1, idx, 1.0)
        soft = (hard_one - soft).detach() + soft
    return soft


class TripleGenerator(nn.Module):
    """Dai encoder + Pix2Pix-style latent injection + three Gumbel-Softmax heads."""
    def __init__(self, num_entities, num_relations, d, d_z,
                 enc_base=32, enc_layers=3, n_heads=4, hidden=128):
        super().__init__()
        self.num_entities  = num_entities
        self.num_relations = num_relations
        self.d   = d
        self.d_z = d_z

        # Dai's 1x1 Conv2d encoder stack on the (1, 3, d) "image" view
        conv_layers = []
        in_c, out_c = 1, enc_base
        for _ in range(enc_layers):
            conv_layers.append(nn.Conv2d(in_c, out_c, kernel_size=(1, 1)))
            conv_layers.append(nn.LeakyReLU(leaky_relu_negative_slope, inplace=True))
            in_c, out_c = out_c, out_c * 2
        self.encoder_conv = nn.Sequential(*conv_layers)
        encoder_final_c = in_c

        self.token_proj = nn.Linear(encoder_final_c * d, hidden)
        self.self_attn  = nn.MultiheadAttention(hidden, n_heads, batch_first=True)
        self.attn_norm  = nn.LayerNorm(hidden)
        self.bottleneck_proj = nn.Linear(hidden * 3 + d_z, hidden)

        self.head_h = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, num_entities))
        self.head_r = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, num_relations))
        self.head_t = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, num_entities))

    def forward(self, clean_triple_emb, z, tau=1.0, return_soft=True):
        B = clean_triple_emb.shape[0]
        x = clean_triple_emb.unsqueeze(1)          # (B, 1, 3, d)
        x = self.encoder_conv(x)                   # (B, C_final, 3, d)
        x = x.permute(0, 2, 1, 3).contiguous().view(B, 3, -1)
        tokens = self.token_proj(x)
        attn_out, _ = self.self_attn(tokens, tokens, tokens)
        tokens = self.attn_norm(tokens + attn_out)
        bn_in = torch.cat([tokens.flatten(1), z], dim=1)
        h_state = self.bottleneck_proj(bn_in)

        h_logits = self.head_h(h_state)
        r_logits = self.head_r(h_state)
        t_logits = self.head_t(h_state)
        out = {'h_logits': h_logits, 'r_logits': r_logits, 't_logits': t_logits}
        if return_soft:
            out['h_soft'] = gumbel_softmax_sample(h_logits, tau)
            out['r_soft'] = gumbel_softmax_sample(r_logits, tau)
            out['t_soft'] = gumbel_softmax_sample(t_logits, tau)
        return out


class TripleDiscriminator(nn.Module):
    """Pix2Pix-style conditional cGAN critic with spectral_norm."""
    def __init__(self, d, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.utils.spectral_norm(nn.Linear(6 * d, hidden)),
            nn.LeakyReLU(leaky_relu_negative_slope, inplace=True),
            nn.utils.spectral_norm(nn.Linear(hidden, hidden // 2)),
            nn.LeakyReLU(leaky_relu_negative_slope, inplace=True),
            nn.utils.spectral_norm(nn.Linear(hidden // 2, 1)),
        )
    def forward(self, clean_triple_emb, candidate_triple_emb):
        x = torch.cat([clean_triple_emb.flatten(1),
                       candidate_triple_emb.flatten(1)], dim=1)
        return self.net(x)


def re_embed_soft(h_soft, r_soft, t_soft, E_weight, R_weight):
    """Gumbel-soft samples -> triple embedding (B, 3, d). Keeps gradients alive."""
    return torch.stack([h_soft @ E_weight, r_soft @ R_weight, t_soft @ E_weight], dim=1)


# === Instantiate ===
entity_embedding   = EntityEmbedding(num_nodes, embedding_dim).to(compute_device)
relation_embedding = RelationEmbedding(num_channels, embedding_dim).to(compute_device)

generator_model = TripleGenerator(
    num_entities=num_nodes,
    num_relations=num_channels,
    d=embedding_dim, d_z=latent_dim,
    enc_base=encoder_base_depth, enc_layers=encoder_n_layers,
    n_heads=encoder_n_heads, hidden=hidden_dim,
).to(compute_device)

discriminator_model = TripleDiscriminator(d=embedding_dim, hidden=256).to(compute_device)

print(f"Generator parameters:    {sum(p.numel() for p in generator_model.parameters()):,}")
print(f"Discriminator parameters: {sum(p.numel() for p in discriminator_model.parameters()):,}")
print(f"EntityEmbedding:          {sum(p.numel() for p in entity_embedding.parameters()):,} "
      f"(|E|={num_nodes} x d={embedding_dim})")
print(f"RelationEmbedding:        {sum(p.numel() for p in relation_embedding.parameters()):,} "
      f"(|R|={num_channels} x d={embedding_dim})")


## Step 7: Shape sanity check (forward pass on a tiny batch)


In [ ]:
with torch.no_grad():
    _clean_idx  = clean_triples_idx[:4].to(compute_device)
    _target_idx = corrupted_triples_idx[:4].to(compute_device)

    _h_emb = entity_embedding(_clean_idx[:, 0])
    _r_emb = relation_embedding(_clean_idx[:, 1])
    _t_emb = entity_embedding(_clean_idx[:, 2])
    _clean_triple_emb = torch.stack([_h_emb, _r_emb, _t_emb], dim=1)

    _z = torch.randn(4, latent_dim, device=compute_device)
    _out = generator_model(_clean_triple_emb, _z, tau=gumbel_tau_start, return_soft=True)

    _cand_emb = re_embed_soft(
        _out['h_soft'], _out['r_soft'], _out['t_soft'],
        entity_embedding.lookup_weight(), relation_embedding.lookup_weight(),
    )
    _d_real = discriminator_model(_clean_triple_emb, _clean_triple_emb)
    _d_fake = discriminator_model(_clean_triple_emb, _cand_emb)

print(f"Clean triple emb: {tuple(_clean_triple_emb.shape)}")
print(f"h_logits / r_logits / t_logits: "
      f"{tuple(_out['h_logits'].shape)} / "
      f"{tuple(_out['r_logits'].shape)} / "
      f"{tuple(_out['t_logits'].shape)}")
print(f"Candidate emb:    {tuple(_cand_emb.shape)}")
print(f"D output shape:   {tuple(_d_real.shape)}")
print(f"D(clean,clean):   {_d_real.mean().item():.3f}")
print(f"D(clean,fake):    {_d_fake.mean().item():.3f}")


## Step 8: Train (Pix2Pix-style alternating G/D)

- `L_recon`: CE on three softmax heads against TRIC target (Pix2Pix's L1 -> CE for categorical)
- `L_adv`:   cGAN BCE
- `L_div`:   mode-seeking diversity regulariser (Mao et al. CVPR 2019)

D-step builds embeddings under no_grad (D doesn't update E, R); G-step rebuilds
them WITH gradient so the embeddings learn via G-step.


In [ ]:
total_epochs        = 400
learning_rate       = 1e-4
adam_beta1          = 0.5
adam_beta2          = 0.999
lambda_recon        = 20.0
lambda_adv          = 1.0
lambda_div          = 0.5
REAL_LABEL_VALUE    = 0.9
FAKE_LABEL_VALUE    = 0.0

# Instance noise on D inputs (Sonderby 2016) - linear decay
instance_noise_std_start = 0.1
instance_noise_std_end   = 0.01

bce_logits_loss = nn.BCEWithLogitsLoss().to(compute_device)
ce_loss         = nn.CrossEntropyLoss().to(compute_device)


def add_instance_noise_emb(clean_emb, cand_emb, std):
    if std <= 0:
        return clean_emb, cand_emb
    return (clean_emb + torch.randn_like(clean_emb) * std,
            cand_emb  + torch.randn_like(cand_emb)  * std)


def mode_seeking_loss(c1, c2, z1, z2, eps=1e-6):
    return -(c1 - c2).abs().mean() / ((z1 - z2).abs().mean() + eps)


def build_triple_emb(idx, e_module, r_module):
    return torch.stack([
        e_module(idx[:, 0]),
        r_module(idx[:, 1]),
        e_module(idx[:, 2]),
    ], dim=1)


g_params = (list(generator_model.parameters())
            + list(entity_embedding.parameters())
            + list(relation_embedding.parameters()))
generator_optimizer = optim.Adam(g_params, lr=learning_rate,
                                 betas=(adam_beta1, adam_beta2))
discriminator_optimizer = optim.Adam(
    discriminator_model.parameters(), lr=learning_rate * 0.25,
    betas=(adam_beta1, adam_beta2),
)

generator_adv_loss_history     = []
generator_recon_loss_history   = []
generator_div_loss_history     = []
generator_total_loss_history   = []
discriminator_loss_history     = []
discriminator_output_on_real   = []
discriminator_output_on_fake_before_g_update = []
discriminator_output_on_fake_after_g_update  = []
instance_noise_std_history     = []
gumbel_tau_history             = []

print(f"Training: {total_epochs} epochs, {len(paired_dataloader)} batches/epoch")
print(f"lambda_recon={lambda_recon}, lambda_adv={lambda_adv}, lambda_div={lambda_div}")
print(f"G lr={learning_rate:.1e}, D lr={learning_rate * 0.25:.1e}")
print(f"Instance noise: std {instance_noise_std_start} -> {instance_noise_std_end} (linear decay)")
print(f"Gumbel tau: {gumbel_tau_start} -> {gumbel_tau_end} (anneal first 50% of training)")
print('-' * 70)

generator_model.train()
discriminator_model.train()

for epoch_index in range(total_epochs):
    epoch_d_loss_sum, epoch_d_loss_count = 0.0, 0

    epoch_progress = epoch_index / max(total_epochs - 1, 1)
    current_noise_std = (instance_noise_std_start * (1 - epoch_progress)
                         + instance_noise_std_end * epoch_progress)
    instance_noise_std_history.append(current_noise_std)

    anneal_progress = min(1.0, epoch_index / max(1, total_epochs // 2))
    current_tau = gumbel_tau_start * (1 - anneal_progress) + gumbel_tau_end * anneal_progress
    gumbel_tau_history.append(current_tau)

    for clean_idx, target_idx in paired_dataloader:
        clean_idx  = clean_idx.to(compute_device)
        target_idx = target_idx.to(compute_device)
        B = clean_idx.size(0)

        real_label = torch.full((B, 1), REAL_LABEL_VALUE, device=compute_device, dtype=torch.float)
        fake_label = torch.full((B, 1), FAKE_LABEL_VALUE, device=compute_device, dtype=torch.float)

        # ----- D-step (embeddings detached) ----------------------------------
        discriminator_model.zero_grad()
        with torch.no_grad():
            clean_emb_d  = build_triple_emb(clean_idx, entity_embedding, relation_embedding)
            target_emb_d = build_triple_emb(target_idx, entity_embedding, relation_embedding)
            z_for_d = torch.randn(B, latent_dim, device=compute_device)
            g_out_d = generator_model(clean_emb_d, z_for_d, tau=current_tau, return_soft=True)
            cand_emb_d = re_embed_soft(
                g_out_d['h_soft'], g_out_d['r_soft'], g_out_d['t_soft'],
                entity_embedding.lookup_weight(), relation_embedding.lookup_weight(),
            )

        clean_n_r, target_n_r = add_instance_noise_emb(clean_emb_d, target_emb_d, current_noise_std)
        d_real      = discriminator_model(clean_n_r, target_n_r)
        d_loss_real = bce_logits_loss(d_real, real_label)

        clean_n_f, cand_n_f = add_instance_noise_emb(clean_emb_d, cand_emb_d, current_noise_std)
        d_fake      = discriminator_model(clean_n_f, cand_n_f)
        d_loss_fake = bce_logits_loss(d_fake, fake_label)

        total_d_loss = d_loss_real + d_loss_fake
        total_d_loss.backward()

        mean_d_real_step     = torch.sigmoid(d_real).mean().item()
        mean_d_fake_before_g = torch.sigmoid(d_fake).mean().item()

        torch.nn.utils.clip_grad_norm_(discriminator_model.parameters(), max_norm=5.0)
        discriminator_optimizer.step()
        epoch_d_loss_sum   += total_d_loss.item()
        epoch_d_loss_count += 1

        # ----- G-step (embeddings WITH gradient) -----------------------------
        generator_optimizer.zero_grad()
        clean_emb_g = build_triple_emb(clean_idx, entity_embedding, relation_embedding)
        z_1 = torch.randn(B, latent_dim, device=compute_device)
        z_2 = torch.randn(B, latent_dim, device=compute_device)

        g_out_1 = generator_model(clean_emb_g, z_1, tau=current_tau, return_soft=True)
        g_out_2 = generator_model(clean_emb_g, z_2, tau=current_tau, return_soft=True)

        cand_emb_1 = re_embed_soft(
            g_out_1['h_soft'], g_out_1['r_soft'], g_out_1['t_soft'],
            entity_embedding.lookup_weight(), relation_embedding.lookup_weight(),
        )
        cand_emb_2 = re_embed_soft(
            g_out_2['h_soft'], g_out_2['r_soft'], g_out_2['t_soft'],
            entity_embedding.lookup_weight(), relation_embedding.lookup_weight(),
        )

        l_recon = (ce_loss(g_out_1['h_logits'], target_idx[:, 0])
                   + ce_loss(g_out_1['r_logits'], target_idx[:, 1])
                   + ce_loss(g_out_1['t_logits'], target_idx[:, 2]))
        clean_n_g, cand_n_g = add_instance_noise_emb(clean_emb_g, cand_emb_1, current_noise_std)
        d_fake_for_g = discriminator_model(clean_n_g, cand_n_g)
        l_adv = bce_logits_loss(d_fake_for_g, real_label)
        mean_d_fake_after_g = torch.sigmoid(d_fake_for_g).mean().item()
        l_div = mode_seeking_loss(cand_emb_1, cand_emb_2, z_1, z_2)

        total_g_loss = lambda_recon * l_recon + lambda_adv * l_adv + lambda_div * l_div
        total_g_loss.backward()
        torch.nn.utils.clip_grad_norm_(g_params, max_norm=5.0)
        generator_optimizer.step()

        generator_adv_loss_history.append(l_adv.item())
        generator_recon_loss_history.append(l_recon.item())
        generator_div_loss_history.append(l_div.item())
        generator_total_loss_history.append(total_g_loss.item())
        discriminator_loss_history.append(total_d_loss.item())
        discriminator_output_on_real.append(mean_d_real_step)
        discriminator_output_on_fake_before_g_update.append(mean_d_fake_before_g)
        discriminator_output_on_fake_after_g_update.append(mean_d_fake_after_g)

    if (epoch_index + 1) % 15 == 0 or epoch_index == 0:
        print(f"[{epoch_index + 1:3d}/{total_epochs}]  "
              f"L_D: {epoch_d_loss_sum / max(epoch_d_loss_count, 1):.4f}  "
              f"L_G_adv: {l_adv.item():.4f}  "
              f"L_G_recon: {l_recon.item():.4f}  "
              f"L_G_div: {l_div.item():+.4f}  "
              f"D(real): {mean_d_real_step:.3f}  "
              f"D(fake): {mean_d_fake_before_g:.3f}/{mean_d_fake_after_g:.3f}  "
              f"noise: {current_noise_std:.3f}  tau: {current_tau:.2f}")

print('-' * 70)
print('Training Complete!')


## Step 9: Training diagnostics


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

axes[0].set_title('Adversarial Losses')
axes[0].plot(generator_adv_loss_history, label='G adv',  alpha=0.7)
axes[0].plot(discriminator_loss_history, label='D total', alpha=0.7)
axes[0].set_xlabel('Iterations'); axes[0].set_ylabel('Loss'); axes[0].legend()

axes[1].set_title('G Reconstruction (CE on 3 heads)')
axes[1].plot(generator_recon_loss_history, color='C2', alpha=0.7)
axes[1].set_xlabel('Iterations'); axes[1].set_ylabel('CE(h)+CE(r)+CE(t)')

axes[2].set_title('Diversity Regulariser\n(more negative = more diverse)')
axes[2].plot(generator_div_loss_history, color='C4', alpha=0.7)
axes[2].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Iterations'); axes[2].set_ylabel('-mean|d_cand|/(mean|d_z|+eps)')

axes[3].set_title('Discriminator Outputs on Pairs')
axes[3].plot(discriminator_output_on_real,                 label='D(clean, real)', alpha=0.7)
axes[3].plot(discriminator_output_on_fake_before_g_update, label='D(clean, fake)', alpha=0.7)
axes[3].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Equilibrium')
axes[3].set_xlabel('Iterations'); axes[3].set_ylabel('D output')
axes[3].set_ylim(-0.05, 1.05); axes[3].legend()

plt.tight_layout(); plt.show()


## Step 10: Generate K corrupted KGs (two-stage inference)


In [ ]:
from inference import generate_k_corrupted_kgs

# Clean triples come straight from the loader - no adjacency, no extraction step.
clean_triples_in_kg = list(kg.triples_idx)
print(f"Clean KG has {len(clean_triples_in_kg)} triples across {num_channels} relation channels.")

K_samples = 20
corrupted_kgs = generate_k_corrupted_kgs(
    generator_model, entity_embedding, relation_embedding,
    clean_triples_in_kg,
    K=K_samples, num_corruptions=num_corruptions,
    latent_dim=latent_dim, tau=0.5, device=compute_device,
    rng=np.random.default_rng(7777),
)
print(f"Drew K={K_samples} corrupted KGs (each with {num_corruptions} corruptions).")


## Step 11: Operation-mix report + human-readable sample view


In [ ]:
from evaluation import summarise_operation_mix, print_sample_details

totals = summarise_operation_mix(clean_triples_in_kg, corrupted_kgs, show_first_n=5)
print_sample_details(
    clean_triples_in_kg, corrupted_kgs,
    node_labels=node_labels, relation_names=relation_names,
    num_samples=3,
)


## Step 12: Visualise GAN vs rule output


In [ ]:
import functools
from corruption_strategies import apply_tric_corruption
from inference.two_stage import kg_triples_to_nxnxr
from visualisation import plot_gan_comparison
from kg_data import triples_to_adjacency_tensor

# Reference rule corruption (same strategy as training, triple-level)
rule_rng    = np.random.default_rng(999)
tric_ref_fn = functools.partial(apply_tric_corruption,
                                num_entities=num_nodes,
                                num_relations=num_channels,
                                node_types=node_types,
                                op_weights={'change_relation': 1})
rule_corr_triples, _edits = tric_ref_fn(clean_triples_in_kg, num_corruptions, rule_rng)

# Convert all three views to (N, N, R) for the plotter
clean_nxnxr_for_plot = triples_to_adjacency_tensor(
    kg.triples, kg.entity_id_to_row, kg.relation_id_to_channel,
)
rule_nxnxr = kg_triples_to_nxnxr(rule_corr_triples, num_nodes, num_channels)
gan_nxnxr  = kg_triples_to_nxnxr(corrupted_kgs[0], num_nodes, num_channels)

plot_gan_comparison(
    clean_nxnxr_for_plot, rule_nxnxr, gan_nxnxr,
    node_labels, node_types, relation_names,
    num_corruptions=num_corruptions,
)


## Step 13: Distribution validation

Compare GAN output to the rule across many samples. A trained GAN should:
- shift roughly `num_corruptions` edges per sample,
- have a net edge delta near zero (change_relation conserves edge count),
- emit `change_relation` for the vast majority of corruption events.


In [ ]:
import functools
from corruption_strategies import apply_tric_corruption
from evaluation import validate_distribution

tric_val_fn = functools.partial(apply_tric_corruption,
                                num_entities=num_nodes,
                                num_relations=num_channels,
                                node_types=node_types,
                                op_weights={'change_relation': 1})

dist_stats = validate_distribution(
    generator_model=generator_model,
    entity_embedding=entity_embedding,
    relation_embedding=relation_embedding,
    clean_triples=clean_triples_in_kg,
    rule_corruption_fn=tric_val_fn,
    num_entities=num_nodes,
    num_relations=num_channels,
    num_samples=200,
    num_corruptions=num_corruptions,
    latent_dim=latent_dim,
    tau=0.5,
    device=compute_device,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3))

rule_counts = dist_stats['rule_shifted_counts']
gan_counts  = dist_stats['gan_shifted_counts']
gan_delta   = dist_stats['gan_net_delta']

bin_edges = np.arange(
    min(rule_counts.min(), gan_counts.min()) - 1,
    max(rule_counts.max(), gan_counts.max()) + 2,
)
axes[0].hist(rule_counts, bins=bin_edges, alpha=0.6, label='Rule', color='C0', edgecolor='black')
axes[0].hist(gan_counts,  bins=bin_edges, alpha=0.6, label='GAN',  color='C3', edgecolor='black')
axes[0].set_xlabel('Edges that changed relation channel')
axes[0].set_ylabel('Sample count')
axes[0].set_title('Relation-shift distribution (rule vs GAN)')
axes[0].legend()

delta_bins = np.arange(gan_delta.min() - 1, gan_delta.max() + 2)
axes[1].hist(gan_delta, bins=delta_bins, alpha=0.7, color='C1', edgecolor='black')
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.7, label='Ideal (0)')
axes[1].set_xlabel('Net edge count change (GAN - clean)')
axes[1].set_ylabel('Sample count')
axes[1].set_title('GAN edge balance')
axes[1].legend()

plt.tight_layout()
plt.show()
